In [ ]:
# # python-dotenv → manage API keys

# !pip install pymupdf pdfplumber pytesseract Pillow \
#              langchain langchain-community \
#              sentence-transformers faiss-cpu \
#              python-dotenv openai -q


In [ ]:
# !pip list

In [ ]:
import camelot

tables = camelot.read_pdf(
    r"D:\python scripts\Generative AI\arise_chatbot\data\petitions\tariff_petition_jusnl.pdf",
    pages="16"
)

df = tables[0].df

headers=df.values.tolist()[0]
rows=df.values.tolist()[1:]

print(headers)
print(rows)
pairs = []
for row in rows:
    for header, value in zip(headers, row):
        pairs.append(f"{header}: {value}")

final_text = " | ".join(pairs)

In [ ]:
final_text

In [ ]:
import fitz          # PyMuPDF
import pdfplumber
import json
import os
import io
from pathlib import Path
from PIL import Image

# Verify versions
print(f"PyMuPDF version  : {fitz.version}")
print(f"pdfplumber       : {pdfplumber.__version__}")
print("✅ Core imports OK")

In [ ]:
%pwd

In [ ]:
import os
os.chdir("../")

In [ ]:
# petition path
PDF_PATH = r"D:\python scripts\Generative AI\arise_chatbot\data\petitions\tariff_petition_jusnl.pdf"
# This is what we discussed — document-level metadata
DOC_METADATA = {
    "discom": "JUSNL",
    "document_type": "petition",   # petition / tariff_order / regulation
    "filing_year": "FY26",         # the year the petition covers
    "commission": "JSERC",
    "source_file": PDF_PATH
}

In [ ]:
# Check file exists
if Path(PDF_PATH).exists():
    print(f"✅ PDF found: {PDF_PATH}")
    doc = fitz.open(PDF_PATH)
    print(f"   Total pages: {len(doc)}")
    doc.close()
else:
    print(f"⚠️  PDF not found at: {PDF_PATH}")
    print("   Place your PDF in the data/petitions/ folder and update PDF_PATH above")

In [ ]:
def detect_page_type(page: fitz.Page) -> str:
    """
    Classify a PDF page as 'digital' or 'scanned'.

    Digital  → PyMuPDF can extract readable text
    Scanned  → Page is an image, text extraction returns nothing
    """
    text = page.get_text().strip()
    # print(text)

    # < 50 chars means either scanned or a blank/diagram page
    # Both cases need OCR to be useful
    if len(text) < 50:
        return "scanned"
    return "digital"


# ── TEST: Run classifier on first 10 pages of your PDF ──
if Path(PDF_PATH).exists():
    doc = fitz.open(PDF_PATH)
    print("Page | Type    | Chars extracted")
    print("-" * 40)
    for i in range(min(10, len(doc))):
        page = doc[i]
        ptype = detect_page_type(page)
        chars = len(page.get_text().strip())
        flag = "⚠️ " if ptype == "scanned" else "✅"
        print(f"  {i+1:2d} | {ptype:<8} | {chars:>5} chars  {flag}")
    doc.close()
else:
    print("⚠️  Add your PDF first (Cell 3)")

In [ ]:
def table_to_llm_text(df, table_index: int) -> str:
    """
    Convert camelot dataframe → clean LLM-readable text.
    Format: Table name + Row-by-row header:value pairs
    """
    rows = df.values.tolist()
    # print(rows)

    if len(rows) < 2:
        return ""

    headers = [str(h).strip() for h in rows[0]]
    # print(headers)
    data_rows = rows[1:]

    lines = [f"Table {table_index + 1}:"]
    # print(data_rows)

    for row_num, row in enumerate(data_rows, start=0):
        pairs = []
        for header, value in zip(headers, row):
            value = str(value).strip()
            # print(value)
            if value:  # skip empty cells
                pairs.append(f"{header}: {value}")
        if pairs:
            lines.append(f"  Row {row_num} → {' | '.join(pairs)}")

    return "\n".join(lines)
import re

def merge_text_and_tables(raw_text: str, tables_text: list[str]) -> str:
    """
    Merge paragraph text and tables in correct sequence.
    Uses table title lines as anchor points for insertion.
    """
    # Split text at table title lines e.g. "Table 3 Capital Expenditure..."
    segments = re.split(r'(Table\s+\d+[^\n]*)', raw_text)

    result = []
    table_idx = 0

    for segment in segments:
        # If this segment is a table title
        if re.match(r'Table\s+\d+', segment.strip()):
            # Insert the clean camelot table here
            if table_idx < len(tables_text):
                result.append(f"\n{segment.strip()}")  # keep title
                result.append(tables_text[table_idx])   # insert clean table
                table_idx += 1
        else:
            if segment.strip():
                result.append(segment.strip())

    return "\n\n".join(result)

def extract_digital_page(pdf_path: str, page_num: int) -> dict:
    """
    Extract text via pdfplumber + tables via camelot.
    Sequence preserved — text first, then tables.
    """
    # Text extraction
    with pdfplumber.open(pdf_path) as pdf:
        page = pdf.pages[page_num]

        # Get bounding boxes of all tables
        table_bboxes = [t.bbox for t in page.find_tables()]

        # Filter out table regions from text extraction
        raw_text = page.filter(
            lambda obj: not any(
                obj["x0"] >= bbox[0] and
                obj["top"] >= bbox[1] and
                obj["x1"] <= bbox[2] and
                obj["bottom"] <= bbox[3]
                for bbox in table_bboxes
            )
        ).extract_text() or ""

    # Table extraction
    tables = camelot.read_pdf(
        pdf_path,
        pages=str(page_num + 1),
        flavor="lattice",
        line_scale=40   # use 'stream' if tables have no borders
    )

    tables_as_text = []
    for i, table in enumerate(tables):
        text = table_to_llm_text(table.df, i)
        # print(text)
        if text:
            tables_as_text.append(text)

    print(f"Tables found: {len(tables)}")
    for i, t in enumerate(tables):
        print(f"Table {i+1}: {t.shape} | accuracy: {t.parsing_report}")

    merged_text = merge_text_and_tables(raw_text, tables_as_text)


    return {
        "text"        : merged_text,
        "tables_text" : tables_as_text,   # LLM-readable
        "tables_raw"  : [t.df for t in tables],  # dataframes for debugging
        "page_number" : page_num + 1,
        "page_type"   : "digital"
    }


# ── TEST: Extract a page that likely has a table ──
# Page 16 in JUSNL petition = Table 3 (Capex) — try this page
TEST_PAGE = 15   # 0-indexed → page 16 in PDF

if Path(PDF_PATH).exists():
    result = extract_digital_page(PDF_PATH, TEST_PAGE)
    print(result['text'])  # print first 500 chars of merged text


    # print(f"=== Page {result['page_number']} ===")
    # print(f"Text length  : {len(result['text'])} chars")
    # print(f"Tables found : {len(result['tables_raw'])}")
    # print()
    # print("--- Text preview (first 400 chars) ---")
    # print(result['text'][:])
    # print()
    # if result['tables_text']:
    #     print("--- First table (cleaned for LLM) ---")
    #     print(result['tables_text'])
    # else:
    #     print("No tables found on this page — try a different TEST_PAGE number")
else:
    print("⚠️  Add your PDF first")

In [ ]:
def extract_scanned_page(pdf_path: str, page_num: int) -> dict:
    """
    Extract text from a scanned PDF page using OCR.

    Steps:
    1. Render PDF page as high-res image (300 DPI)
    2. Run Tesseract OCR on the image
    3. Return extracted text

    Note: Tables from scanned pages come out as plain text.
    Structured table extraction from scans needs advanced tools
    like AWS Textract or Google Document AI.
    """
    try:
        import pytesseract
    except ImportError:
        return {"text": "", "tables_text": [], "tables_raw": [],
                "page_number": page_num + 1, "page_type": "scanned",
                "error": "pytesseract not installed"}

    doc = fitz.open(pdf_path)
    page = doc[page_num]

    # 300 DPI gives good OCR quality without being too slow
    # 72 DPI is screen resolution → 300/72 ≈ 4.16x scale
    mat = fitz.Matrix(300 / 72, 300 / 72)
    pix = page.get_pixmap(matrix=mat)
    img = Image.open(io.BytesIO(pix.tobytes("png")))
    doc.close()

    # Run OCR — 'eng' for English, add '+hin' if Hindi text exists
    text = pytesseract.image_to_string(img, lang='eng')

    return {
        "text": text,
        "tables_text": [],    # no structured tables from OCR
        "tables_raw": [],
        "page_number": page_num + 1,
        "page_type": "scanned"
    }


# ── TEST: Only run if you have a scanned page ──
# Find a scanned page number from Stage 1 output above
# then set SCANNED_PAGE to that number (0-indexed)

SCANNED_PAGE = None   # Set this to a scanned page number, e.g. 5

if SCANNED_PAGE is not None and Path(PDF_PATH).exists():
    result = extract_scanned_page(PDF_PATH, SCANNED_PAGE)
    print(f"=== Scanned Page {result['page_number']} ===")
    print(f"OCR text length: {len(result['text'])} chars")
    print("--- OCR output preview ---")
    print(result['text'][:400])
else:
    print("ℹ️  Skipped OCR test — set SCANNED_PAGE to a scanned page number")
    print("   (Run Stage 1 classifier first to find scanned pages)")

In [ ]:
def load_pdf(pdf_path: str, doc_metadata: dict, max_pages: int = None) -> list[dict]:
    """
    Load entire PDF. For each page:
    1. Classify (digital or scanned)
    2. Extract with correct extractor
    3. Attach document-level metadata to every page

    Returns list of page dicts, one per page.
    """
    doc = fitz.open(pdf_path)
    total_pages = len(doc)

    # Optionally limit pages for testing
    pages_to_process = min(max_pages, total_pages) if max_pages else total_pages

    print(f"📄 Loading: {pdf_path}")
    print(f"   Total pages: {total_pages} | Processing: {pages_to_process}")
    print()

    all_pages = []

    for i in range(pages_to_process):
        page = doc[i]
        page_type = detect_page_type(page)

        # Route to correct extractor
        if page_type == "digital":
            page_data = extract_digital_page(pdf_path, i)
        else:
            page_data = extract_scanned_page(pdf_path, i)

        # Merge document-level metadata into every page
        # This is critical — every chunk inherits this later
        page_data.update(doc_metadata)

        all_pages.append(page_data)

        # Progress indicator
        icon = "✅" if page_type == "digital" else "⚠️ "
        print(f"  {icon} Page {i+1:3d}/{pages_to_process} | {page_type:<8} | "
              f"{len(page_data['text']):>5} chars | "
              f"{len(page_data.get('tables_raw', []))} tables")

    doc.close()
    print(f"\n✅ Done — {len(all_pages)} pages loaded")
    return all_pages



PDF_PATH = r"D:\python scripts\Generative AI\arise_chatbot\data\petitions\tariff_petition_jusnl.pdf"


# ── TEST: Load first 20 pages (fast) ──
if Path(PDF_PATH).exists():
    pages = load_pdf(PDF_PATH, DOC_METADATA, max_pages=20)
    print(DOC_METADATA)
    print(f"\nSample page keys: {list(pages[0].keys())}")
else:
    print("⚠️  Add your PDF first")

In [ ]:
import re

# Headings found in Indian regulatory documents
# These signal a new section → natural chunk boundary
SECTION_PATTERNS = [
    r'^\d+\.\d+\s+[A-Z]',          # 3.3 Capital Expenditure
    r'^\d+\.\s+[A-Z]',             # 3. Overall Approach
    r'^[A-Z][A-Z\s]{10,}$',        # ALL CAPS HEADING
    r'^(Chapter|Section|Part)\s+',  # Chapter / Section / Part
]

def is_section_heading(line: str) -> bool:
    """Check if a line looks like a section heading."""
    line = line.strip()
    if len(line) < 5 or len(line) > 120:
        return False
    return any(re.match(p, line) for p in SECTION_PATTERNS)


def chunk_text(text: str, page_metadata: dict) -> list[dict]:
    """
    Split page text into chunks at section boundaries.
    Each chunk inherits the page's metadata.
    """
    lines = text.split('\n')
    chunks = []
    current_chunk_lines = []
    current_heading = "introduction"

    for line in lines:
        if is_section_heading(line):
            # Save current chunk before starting new section
            if current_chunk_lines:
                chunk_text = '\n'.join(current_chunk_lines).strip()
                if len(chunk_text) > 100:  # skip tiny fragments
                    chunks.append({
                        "text": chunk_text,
                        "chunk_type": "text",
                        "section_heading": current_heading,
                        **page_metadata   # inherit all page metadata
                    })
            # Start new section
            current_heading = line.strip()
            current_chunk_lines = [line]
        else:
            current_chunk_lines.append(line)

    # Don't forget the last chunk
    if current_chunk_lines:
        chunk_text_val = '\n'.join(current_chunk_lines).strip()
        if len(chunk_text_val) > 100:
            chunks.append({
                "text": chunk_text_val,
                "chunk_type": "text",
                "section_heading": current_heading,
                **page_metadata
            })

    return chunks


def chunk_tables(tables_text: list[str], page_metadata: dict) -> list[dict]:
    """
    Each table becomes its own dedicated chunk.
    Tables are NEVER split — they lose meaning if split.
    """
    chunks = []
    for i, table_text in enumerate(tables_text):
        if table_text.strip():
            chunks.append({
                "text": table_text,
                "chunk_type": "table",      # important for retrieval later
                "table_index": i,
                "section_heading": "table",
                **page_metadata
            })
    return chunks


def chunk_page(page_data: dict) -> list[dict]:
    """
    Chunk a single page's content.
    Returns list of chunks (text chunks + table chunks).
    """
    # Metadata to pass down to every chunk from this page
    page_metadata = {
        "page_number" : page_data["page_number"],
        "page_type"   : page_data["page_type"],
        "discom"      : page_data.get("discom", ""),
        "document_type": page_data.get("document_type", ""),
        "filing_year" : page_data.get("filing_year", ""),
        "commission"  : page_data.get("commission", ""),
        "source_file" : page_data.get("source_file", "")
    }

    all_chunks = []

    # Chunk the text
    if page_data.get("text"):
        text_chunks = chunk_text(page_data["text"], page_metadata)
        all_chunks.extend(text_chunks)

    # Chunk the tables (each table = one chunk)
    if page_data.get("tables_text"):
        table_chunks = chunk_tables(page_data["tables_text"], page_metadata)
        all_chunks.extend(table_chunks)

    return all_chunks


# ── TEST: Chunk all loaded pages ──
if 'pages' in dir() and pages:
    all_chunks = []
    for page_data in pages:
        chunks = chunk_page(page_data)
        all_chunks.extend(chunks)

    # Summary
    text_chunks  = [c for c in all_chunks if c['chunk_type'] == 'text']
    table_chunks = [c for c in all_chunks if c['chunk_type'] == 'table']

    print(f"Total chunks  : {len(all_chunks)}")
    print(f"Text chunks   : {len(text_chunks)}")
    print(f"Table chunks  : {len(table_chunks)}")
    print()
    print("--- Sample text chunk ---")
    if text_chunks:
        c = text_chunks[0]
        print(f"  Page       : {c['page_number']}")
        print(f"  Heading    : {c['section_heading']}")
        print(f"  Discom     : {c['discom']}")
        print(f"  Text preview: {c['text'][:200]}")
    print()
    print("--- Sample table chunk ---")
    if table_chunks:
        c = table_chunks[0]
        print(f"  Page       : {c['page_number']}")
        print(f"  Table index: {c['table_index']}")
        print(f"  Content    : {c['text'][:300]}")
else:
    print("⚠️  Load PDF first (Stage 4)")

In [ ]:
# Option A: Use OpenAI (needs API key)
# Option B: Use a free local approach with keyword matching first,
#           then LLM only for ambiguous cases
#
# We start with Option B — no API key needed to learn

# Standard cost heads from Indian electricity regulations
COST_HEAD_KEYWORDS = {
    "capex"         : ["capital expenditure", "capex", "cwip", "capitalization",
                       "capital investment", "scheme", "project"],
    "employee_cost" : ["employee", "salary", "staff", "manpower", "hr",
                       "wages", "pension", "gratuity"],
    "om_expenses"   : ["operation and maintenance", "o&m", "repair",
                       "maintenance", "a&g", "administrative"],
    "depreciation"  : ["depreciation", "asset life", "useful life", "dep rate"],
    "interest"      : ["interest", "loan", "borrowing", "debt", "working capital",
                       "interest on loan"],
    "roe"           : ["return on equity", "roe", "equity", "normative equity"],
    "power_purchase": ["power purchase", "ppc", "energy charge", "capacity charge",
                       "ppa", "merit order", "short term purchase"],
    "revenue_gap"   : ["revenue gap", "regulatory asset", "carrying cost",
                       "unrecovered", "revenue requirement"],
    "tariff"        : ["tariff", "consumer category", "slab", "transmission charge",
                       "wheeling charge"],
    "true_up"       : ["true up", "true-up", "trueup", "actual vs approved",
                       "reconciliation", "provisional"],
    "arr"           : ["annual revenue requirement", "arr", "aggregate revenue"],
    "at_c_loss"     : ["at&c", "aggregate technical", "commercial loss", "transmission loss"]
}

def map_cost_head_keywords(text: str, heading: str) -> str:
    """
    Fast keyword-based cost head mapping.
    Check heading first (more reliable), then full text.
    Returns matched cost head or 'other'.
    """
    search_text = (heading + " " + text[:500]).lower()

    scores = {}
    for cost_head, keywords in COST_HEAD_KEYWORDS.items():
        score = sum(1 for kw in keywords if kw in search_text)
        if score > 0:
            scores[cost_head] = score

    if scores:
        return max(scores, key=scores.get)  # return highest scoring cost head
    return "other"


def add_metadata_to_chunks(chunks: list[dict]) -> list[dict]:
    """
    Add cost_head to every chunk.
    Uses keyword matching (fast, free).
    Later we'll upgrade this to LLM mapping for ambiguous cases.
    """
    for chunk in chunks:
        cost_head = map_cost_head_keywords(
            chunk.get("text", ""),
            chunk.get("section_heading", "")
        )
        chunk["cost_head"] = cost_head
    return chunks


# ── TEST: Add cost_head metadata to all chunks ──
if 'all_chunks' in dir() and all_chunks:
    all_chunks = add_metadata_to_chunks(all_chunks)

    # Show distribution of cost heads
    from collections import Counter
    cost_head_counts = Counter(c['cost_head'] for c in all_chunks)

    print("Cost head distribution across chunks:")
    print("-" * 35)
    for cost_head, count in cost_head_counts.most_common():
        bar = "█" * count
        print(f"  {cost_head:<18} {count:>3}  {bar}")

    print()
    print("--- Sample chunk with full metadata ---")
    sample = next((c for c in all_chunks if c['cost_head'] == 'capex'), all_chunks[0])
    meta_only = {k: v for k, v in sample.items() if k != 'text'}
    print(json.dumps(meta_only, indent=2))
    print(f"  text preview: {sample['text'][:150]}")
else:
    print("⚠️  Run Stage 5 first")

In [ ]:
from langchain_groq import ChatGroq
from langchain.schema import HumanMessage
import os
import json

client = ChatGroq(
    groq_api_key=os.getenv("GROQ_API_KEY"),
    model_name="llama-3.3-70b-versatile",
    temperature=0
)

COST_HEADS = [
    "capex", "employee_cost", "om_expenses", "depreciation",
    "interest", "roe", "power_purchase", "revenue_gap",
    "tariff", "true_up", "arr", "at_c_loss", "other"
]

def map_cost_head_llm(text: str, heading: str) -> str:

    prompt = f"""
You are an Indian power sector regulatory expert.

Given this text from a tariff petition:

HEADING: {heading}

TEXT:
{text[:400]}

Which ONE cost head does this belong to?

Choose exactly one from:
{", ".join(COST_HEADS)}

Reply with only the cost head name.
"""

    response = client.invoke([
        HumanMessage(content=prompt)
    ])

    result = response.content.strip().lower()

    if result in COST_HEADS:
        return result

    return "other"


def add_metadata_to_chunks(chunks):

    print(f"Mapping {len(chunks)} chunks to cost heads...")

    for i, chunk in enumerate(chunks):

        cost_head = map_cost_head_llm(
            chunk.get("text", ""),
            chunk.get("section_heading", "")
        )

        chunk["cost_head"] = cost_head

        print(f"{i+1}/{len(chunks)} → {cost_head}")

    return chunks

if 'all_chunks' in dir() and all_chunks:
    all_chunks = add_metadata_to_chunks(all_chunks)

    # Show distribution of cost heads
    from collections import Counter
    cost_head_counts = Counter(c['cost_head'] for c in all_chunks)

    print("Cost head distribution across chunks:")
    print("-" * 35)
    for cost_head, count in cost_head_counts.most_common():
        bar = "█" * count
        print(f"  {cost_head:<18} {count:>3}  {bar}")

    print()
    print("--- Sample chunk with full metadata ---")
    sample = next((c for c in all_chunks if c['cost_head'] == 'capex'), all_chunks[0])
    meta_only = {k: v for k, v in sample.items() if k != 'text'}
    print(json.dumps(meta_only, indent=2))
    print(f"  text preview: {sample['text'][:150]}")
else:
    print("⚠️  Run Stage 5 first")